# BDG2 Federated Split + Feature-Asymmetric Deployment — Paper 3

Turns each shortlisted building into a federated **client** with:
- engineered features (cyclic calendar + weather + causal lags/rolling means)
- chronological **TRAIN / VAL / CALIB / TEST** splits (CALIB = the held-out set for conformal / post-hoc quantiles)
- a **feature-asymmetric deployment** condition: at CALIB/TEST only `{calendar, air-temperature}` are live; the other weather channels are reconstructed from train-only estimators (mirrors the V/I/P^PV reconstruction of Papers 1-2).

Each client is saved separately (data never pooled) plus a `manifest.json`.

**Run `bdg2_profile.ipynb` first** so that `bdg2_profile_out/candidate_clients.csv` exists.

## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Imports, paths, config

In [2]:
import os, sys, json
import numpy as np
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/paper3_fed_load"
DATA_DIR    = os.path.join(PROJECT_DIR, "data")
CLIENTS_CSV = os.path.join(PROJECT_DIR, "bdg2_profile_out", "candidate_clients.csv")
OUT_DIR     = os.path.join(PROJECT_DIR, "bdg2_federated_out")
CLI_DIR     = os.path.join(OUT_DIR, "clients")
os.makedirs(CLI_DIR, exist_ok=True)

# ---- config ----
TARGET_METER = "electricity"
USE_CLEANED  = True
SPLIT_FRACS  = {"train": 0.65, "val": 0.10, "calib": 0.10, "test": 0.15}
LAGS         = [1, 2, 24]
ROLLS        = [3, 6, 24]
DEGRADED_CHANNELS = ["dewTemperature", "cloudCoverage", "precipDepth1HR",
                     "seaLvlPressure", "windSpeed", "windDirection"]
CLUSTER_BY   = "primary_use"   # None, "primary_use", or "site_id"

## 3. Helper functions

In [15]:
def find_col(df, cands):
    lut = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in lut:
            return lut[c.lower()]
    return None


def load_inputs():
    sub = "cleaned" if USE_CLEANED else "raw"
    suf = "_cleaned" if USE_CLEANED else ""
    mp = os.path.join(DATA_DIR, "meters", sub, f"{TARGET_METER}{suf}.csv")
    wp = os.path.join(DATA_DIR, "weather", "weather.csv")
    dp = os.path.join(DATA_DIR, "metadata", "metadata.csv")
    for p in (mp, wp, dp):
        if not os.path.exists(p):
            sys.exit(f"[ERROR] missing {p}")
        if os.path.getsize(p) < 2000 and "git-lfs" in open(p).read(200):
            sys.exit(f"[ERROR] {p} is a Git-LFS pointer. Run `git lfs pull` or download from Zenodo (10.5281/zenodo.3887306).")
    return pd.read_csv(mp), pd.read_csv(wp), pd.read_csv(dp)


def add_calendar(df):
    idx = df.index
    hod, dow, doy = idx.hour, idx.dayofweek, idx.dayofyear
    tp = 2 * np.pi
    df["hod_sin"] = np.sin(tp * hod / 24);  df["hod_cos"] = np.cos(tp * hod / 24)
    df["dow_sin"] = np.sin(tp * dow / 7);   df["dow_cos"] = np.cos(tp * dow / 7)
    df["doy_sin"] = np.sin(tp * doy / 365); df["doy_cos"] = np.cos(tp * doy / 365)
    return df


def add_causal(df):
    for l in LAGS:
        df[f"load_lag{l}"] = df["load"].shift(l)
    for w in ROLLS:
        df[f"load_ma{w}"] = df["load"].shift(1).rolling(w).mean()
    return df


def split_labels(n):
    b_tr = int(n * SPLIT_FRACS["train"])
    b_va = b_tr + int(n * SPLIT_FRACS["val"])
    b_ca = b_va + int(n * SPLIT_FRACS["calib"])
    lab = np.empty(n, dtype=object)
    lab[:b_tr] = "train"; lab[b_tr:b_va] = "val"
    lab[b_va:b_ca] = "calib"; lab[b_ca:] = "test"
    return lab


def fit_reconstructors(train_df, channels):
    """Fit contextual-mean and calendar+airtemp regression on TRAIN; pick lower error per channel."""
    recon = {}
    def design(d):
        return np.column_stack([np.ones(len(d)), d["airTemperature"].fillna(0),
                                d["hod_sin"], d["hod_cos"], d["dow_sin"], d["dow_cos"]])
    for ch in channels:
        if ch not in train_df or train_df[ch].notna().sum() < 50:
            continue
        tr = train_df.dropna(subset=[ch])
        key = [tr.index.hour, tr.index.dayofweek]
        ctx = tr.groupby(key)[ch].mean(); g = tr[ch].mean()
        def ctx_pred(d, ctx=ctx, g=g):
            keys = list(zip(d.index.hour, d.index.dayofweek))
            return np.array([ctx.get(k, g) for k in keys])
        beta, *_ = np.linalg.lstsq(design(tr), tr[ch].values, rcond=None)
        def lin_pred(d, beta=beta):
            return design(d) @ beta
        e_ctx = np.mean(np.abs(ctx_pred(tr) - tr[ch].values))
        e_lin = np.mean(np.abs(lin_pred(tr) - tr[ch].values))
        recon[ch] = ("contextual_mean", ctx_pred) if e_ctx <= e_lin else ("calendar_regression", lin_pred)
    return recon


def build_client(bid, meters, ts_col, weather, wcols, site_of):
    s = meters[[ts_col, bid]].rename(columns={bid: "load"}).dropna(subset=[ts_col])
    s[ts_col] = pd.to_datetime(s[ts_col])
    s = s.set_index(ts_col).sort_index()
    s = s[~s.index.duplicated(keep="first")]                    # de-dup meter timestamps
    full = pd.date_range(s.index.min(), s.index.max(), freq="h")  # 'h' silences the warning
    s = s.reindex(full)
    site = site_of.get(bid)
    if site is not None and wcols:
        w = weather[weather["__site__"] == site].set_index("__ts__").sort_index()
        w = w[~w.index.duplicated(keep="first")]                # de-dup weather timestamps
        w = w[[c for c in wcols if c in w.columns]].reindex(full).interpolate(limit=6)
        s = s.join(w)
    s = add_calendar(s)
    s = add_causal(s)
    s["split"] = split_labels(len(s))
    return s

## 4. Load inputs and normalise weather / metadata handles

In [16]:
if not os.path.exists(CLIENTS_CSV):
    sys.exit(f"[ERROR] {CLIENTS_CSV} not found. Run bdg2_profile.ipynb first.")
clients = pd.read_csv(CLIENTS_CSV)
client_ids = clients["building_id"].tolist()
print(f"{len(client_ids)} client buildings from shortlist")

meters, weather, meta = load_inputs()
ts_col = find_col(meters, ["timestamp", "time", "datetime"])

w_site = find_col(weather, ["site_id", "site"])
w_ts   = find_col(weather, ["timestamp", "time"])
weather["__site__"] = weather[w_site]
weather["__ts__"]   = pd.to_datetime(weather[w_ts])
airt = find_col(weather, ["airTemperature", "temperature"])
if airt and airt != "airTemperature":
    weather = weather.rename(columns={airt: "airTemperature"})
wcols = [c for c in (["airTemperature"] + DEGRADED_CHANNELS) if c in weather.columns]

m_bid  = find_col(meta, ["building_id", "building"])
m_site = find_col(meta, ["site_id", "site"])
m_use  = find_col(meta, ["primaryspaceusage", "primary_use"])
site_of = dict(zip(meta[m_bid], meta[m_site])) if m_site else {}
use_of  = dict(zip(meta[m_bid], meta[m_use]))  if m_use else {}
print("weather channels available:", wcols)

60 client buildings from shortlist
weather channels available: ['airTemperature', 'dewTemperature', 'cloudCoverage', 'precipDepth1HR', 'seaLvlPressure', 'windSpeed', 'windDirection']


## 5. Build each client (split + feature-asymmetric reconstruction)

In [17]:
manifest = {"target_meter": TARGET_METER, "use_cleaned": USE_CLEANED,
            "split_fracs": SPLIT_FRACS, "lags": LAGS, "rolls": ROLLS,
            "known_future": ["calendar(hod,dow,doy cyc)", "airTemperature"],
            "degraded_at_deploy": [c for c in DEGRADED_CHANNELS if c in wcols],
            "causal_seeded_then_rolled": [f"load_lag{l}" for l in LAGS] + [f"load_ma{w}" for w in ROLLS],
            "clients": {}}

try:
    import pyarrow  # noqa
    use_pq = True
except Exception:
    use_pq = False

for bid in client_ids:
    if bid not in meters.columns:
        print(f"[skip] {bid} not in meter file")
        continue
    df = build_client(bid, meters, ts_col, weather, wcols, site_of)

    deg = [c for c in DEGRADED_CHANNELS if c in df.columns]
    recon = fit_reconstructors(df[df["split"] == "train"], deg)
    choices = {}
    mask_deploy = df["split"].isin(["calib", "test"])
    for ch, (name, fn) in recon.items():
        df[f"{ch}__observed"] = df[ch]
        rec = df[ch].copy()
        rec.loc[mask_deploy] = fn(df.loc[mask_deploy])
        df[f"{ch}__reconstructed"] = rec
        choices[ch] = name

    path = os.path.join(CLI_DIR, f"{bid}.parquet" if use_pq else f"{bid}.csv.gz")
    (df.to_parquet(path) if use_pq else df.to_csv(path, compression="gzip"))
    manifest["clients"][bid] = {
        "site": site_of.get(bid),
        "cluster": (use_of.get(bid) if CLUSTER_BY == "primary_use" else site_of.get(bid)),
        "n_rows": int(len(df)),
        "date_start": str(df.index.min()), "date_end": str(df.index.max()),
        "n_train": int((df.split == "train").sum()),
        "n_val":   int((df.split == "val").sum()),
        "n_calib": int((df.split == "calib").sum()),
        "n_test":  int((df.split == "test").sum()),
        "reconstruction_choice": choices,
        "file": os.path.basename(path)}
    print(f"[ok] {bid:35s} rows={len(df):6d}  recon={choices}")

with open(os.path.join(OUT_DIR, "manifest.json"), "w") as fh:
    json.dump(manifest, fh, indent=2, default=str)
print(f"\nwrote {len(manifest['clients'])} clients to {CLI_DIR}")

[ok] Hog_education_Josh                  rows= 17544  recon={'dewTemperature': 'calendar_regression', 'cloudCoverage': 'contextual_mean', 'precipDepth1HR': 'contextual_mean', 'seaLvlPressure': 'calendar_regression', 'windSpeed': 'contextual_mean', 'windDirection': 'calendar_regression'}
[ok] Robin_education_Karyl               rows= 17544  recon={'dewTemperature': 'calendar_regression', 'cloudCoverage': 'contextual_mean', 'seaLvlPressure': 'contextual_mean', 'windSpeed': 'calendar_regression', 'windDirection': 'calendar_regression'}
[ok] Robin_office_Victor                 rows= 17544  recon={'dewTemperature': 'calendar_regression', 'cloudCoverage': 'contextual_mean', 'seaLvlPressure': 'contextual_mean', 'windSpeed': 'calendar_regression', 'windDirection': 'calendar_regression'}
[ok] Lamb_education_Emilie               rows= 17544  recon={'dewTemperature': 'calendar_regression', 'cloudCoverage': 'contextual_mean', 'windSpeed': 'contextual_mean', 'windDirection': 'calendar_regression'}


## 6. Cluster summary + sanity check

In [18]:
if CLUSTER_BY:
    clus = pd.Series({b: v["cluster"] for b, v in manifest["clients"].items()})
    print("clients per cluster:")
    print(clus.value_counts().to_string())

print("\nEach client file columns:")
print("  load, <weather>, <weather>__observed, <weather>__reconstructed,")
print("  calendar(hod/dow/doy sin&cos), load_lag*, load_ma*, split")
print("\nToggle observed vs __reconstructed at deploy to run the asymmetry test.")

# peek at one client
import glob
f0 = sorted(glob.glob(os.path.join(CLI_DIR, "*")))[0]
print("\nexample client:", os.path.basename(f0))
(pd.read_parquet(f0) if f0.endswith(".parquet") else pd.read_csv(f0)).head()

clients per cluster:
Education    30
Office       30

Each client file columns:
  load, <weather>, <weather>__observed, <weather>__reconstructed,
  calendar(hod/dow/doy sin&cos), load_lag*, load_ma*, split

Toggle observed vs __reconstructed at deploy to run the asymmetry test.

example client: Bear_education_Bob.parquet


,load,airTemperature,dewTemperature,cloudCoverage,precipDepth1HR,seaLvlPressure,windSpeed,windDirection,hod_sin,hod_cos,...,cloudCoverage__observed,cloudCoverage__reconstructed,precipDepth1HR__observed,precipDepth1HR__reconstructed,seaLvlPressure__observed,seaLvlPressure__reconstructed,windSpeed__observed,windSpeed__reconstructed,windDirection__observed,windDirection__reconstructed
2016-01-01 00:00:00,75.2022,4.4,-2.2,0.0,0.0,1020.9,0.0,0.0,0.000000,1.000000,...,0.0,0.0,0.0,0.0,1020.9,1020.9,0.0,0.0,0.0,0.0
2016-01-01 01:00:00,84.1075,4.4,-4.4,0.0,0.0,1020.5,2.1,20.0,0.258819,0.965926,...,0.0,0.0,0.0,0.0,1020.5,1020.5,2.1,2.1,20.0,20.0
2016-01-01 02:00:00,79.9268,4.4,-6.7,0.0,0.0,1020.8,2.1,20.0,0.500000,0.866025,...,0.0,0.0,0.0,0.0,1020.8,1020.8,2.1,2.1,20.0,20.0
2016-01-01 03:00:00,76.5400,4.4,-7.8,0.0,0.0,1020.7,2.6,30.0,0.707107,0.707107,...,0.0,0.0,0.0,0.0,1020.7,1020.7,2.6,2.6,30.0,30.0
2016-01-01 04:00:00,75.9115,5.0,-9.4,0.0,0.0,1020.6,0.0,0.0,0.866025,0.500000,...,0.0,0.0,0.0,0.0,1020.6,1020.6,0.0,0.0,0.0,0.0


In [19]:
import pandas as pd, glob, os
f = sorted(glob.glob(os.path.join(CLI_DIR, "*.parquet")))[0]
d = pd.read_parquet(f)
print("split sizes:", d["split"].value_counts().to_dict())
te = d[d["split"] == "test"]
for ch in ["dewTemperature", "cloudCoverage", "seaLvlPressure", "windSpeed"]:
    if f"{ch}__observed" in d:
        diff = (te[f"{ch}__observed"] - te[f"{ch}__reconstructed"]).abs().mean()
        print(f"{ch:16s} mean|obs-recon| at TEST = {diff:.3f}")

split sizes: {'train': 11403, 'test': 2633, 'val': 1754, 'calib': 1754}
dewTemperature   mean|obs-recon| at TEST = 2.946
cloudCoverage    mean|obs-recon| at TEST = 1.213
seaLvlPressure   mean|obs-recon| at TEST = 3.020
windSpeed        mean|obs-recon| at TEST = 1.848
